# Feature Engineering

This notebook engineers rich features from the preprocessed datasets.

**Inputs** (`data/processed/`):
- `movies_integrated.csv` -- from `03_data_preprocessing.ipynb`
- `train_ratings.csv`, `test_ratings.csv` -- user-based 80/20 split

**Outputs** (`data/processed/`):
- `movie_engineered.csv` -- movie-level features (44,641 x 15)
- `user_engineered.csv` -- user-level features (256,107 x 10)
- `movie_features_selected.csv` -- after selection (44,641 x 11)
- `user_features_selected.csv` -- after selection (256,107 x 8)

**Next:** `05_popularity_baseline.ipynb`

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

from src.data.data_loader import DataLoader
from src.data.data_preprocessor import DataPreprocessor
from src.features.feature_engineering import FeatureEngineer
from src.features.feature_selection import FeatureSelector

print("Libraries loaded.")

Libraries loaded.


## Load Preprocessed Data

In [2]:
loader = DataLoader()

print("Loading preprocessed data...")
movies, train_ratings, test_ratings = loader.load_preprocessed_data()
print(f"  movies_integrated: {movies.shape}")
print(f"  train_ratings:     {train_ratings.shape}")
print(f"  test_ratings:      {test_ratings.shape}")


Loading preprocessed data...
  movies_integrated: (43549, 59)
  train_ratings:     (20881168, 10)
  test_ratings:      (5109682, 10)


## Engineer Movie Features

Movie-level features computed from the training set:
- **Popularity**: Bayesian weighted rating, vote-count bins, rating percentile
- **Temporal**: release year, decade, recency score
- **Quality**: avg_rating, std_rating, rating consistency
- **Engagement**: n_ratings, n_unique_users, popularity_rank


In [3]:
engineer = FeatureEngineer()

print("Engineering movie features...")
movie_features = engineer.engineer_movie_features(
    train_ratings=train_ratings,
    movies_integrated=movies,
)
print(f"  movie_features shape: {movie_features.shape}")
print(f"  Columns: {list(movie_features.columns[:10])} ... ({len(movie_features.columns)} total)")

Engineering movie features...
  movie_features shape: (44641, 15)
  Columns: ['movieId', 'n_ratings', 'avg_rating', 'std_rating', 'min_rating', 'max_rating', 'n_unique_users', 'rating_consistency', 'popularity_rank', 'avg_rating_rank'] ... (15 total)


## Engineer User Features

User-level features computed from the training set:
- **Activity**: total ratings, n_unique_movies, rating_diversity
- **Taste**: avg_rating, std_rating, rating_bias
- **Genre preferences**: top_genres (weighted by ratings given)
- **Behaviour**: min/max rating spread


In [4]:
print("Engineering user features (this may take a few minutes for 162K users)...")
user_features = engineer.engineer_user_features(
    train_ratings=train_ratings,
    movies_integrated=movies,
)
print(f"  user_features shape: {user_features.shape}")
print(f"  Columns: {list(user_features.columns[:10])} ... ({len(user_features.columns)} total)")

Engineering user features (this may take a few minutes for 162K users)...
  user_features shape: (256107, 10)
  Columns: ['userId', 'n_ratings', 'avg_rating', 'std_rating', 'min_rating', 'max_rating', 'n_unique_movies', 'rating_diversity', 'top_genres', 'user_rating_bias'] ... (10 total)


## Feature Selection

Reduce dimensionality using variance thresholding (removes near-constant features)
and correlation filtering (removes redundant pairs with r > 0.95).


In [5]:
selector = FeatureSelector()

print("Selecting movie features...")
movie_features_selected = selector.select_movie_features(movie_features)
print(f"  {movie_features.shape[1]} -> {movie_features_selected.shape[1]} features")

print("Selecting user features...")
user_features_selected = selector.select_user_features(user_features)
print(f"  {user_features.shape[1]} -> {user_features_selected.shape[1]} features")


Selecting movie features...
  15 -> 11 features
Selecting user features...
  10 -> 8 features


## Save Engineered Features

In [6]:
engineer.save_engineered_features(
    user_engineered=user_features,
    movie_engineered=movie_features,
)
engineer.save_features(
    user_features=user_features_selected,
    movie_features=movie_features_selected,
)
print("Feature files saved to data/processed/")
print("Next: run 05_popularity_baseline.ipynb")

Engineered features saved successfully!
Engineered features saved successfully!
Feature files saved to data/processed/
Next: run 05_popularity_baseline.ipynb


## Summary

| Feature set | Before selection | After selection | Rows |
|---|---|---|---|
| Movie features | 15 columns | 11 columns | 44,641 |
| User features | 10 columns | 8 columns | 256,107 |

All four files saved to `data/processed/`. The selected features are used
by modelling notebooks 05 through 08.

**Next:** `05_popularity_baseline.ipynb`